# Advanced: Nonlinear Optimization
In [the L9c lecture](CHEME-5800-L9c-Lecture-LogisticRegressionAndOptimization-Fall-2026.ipynb), we minimized the cross-entropy loss with gradient descent, and we handled constraints in simulated annealing by adding penalty and barrier terms to the objective. In this advanced notebook, we look at the theory behind constrained problems, the conditions that a solution must satisfy, and how gradient descent can be adapted to handle the constraints and made faster.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
>
> * __State the KKT conditions for a constrained nonlinear problem:__ The Karush–Kuhn–Tucker (KKT) conditions generalize the method of Lagrange multipliers to problems with both equality and inequality constraints. We'll write the Lagrangian of the general problem and the four conditions that an optimal solution must satisfy.
> * __Explain when the KKT conditions are necessary and when they are sufficient:__ A condition on the constraints, such as Slater's condition, makes the KKT conditions necessary for optimality, and convexity makes them sufficient. We'll work through a small example where the constraint is active in one case and inactive in the other.
> * __Adapt gradient descent to constrained problems:__ Gradient descent is designed for problems without constraints. We'll fold the constraints into the objective with barrier and penalty terms, work through the resulting algorithm, and look at variants that speed up gradient descent.

Let's get started!
___

## General Problem
Let's begin with a general nonlinear optimization problem and the theory needed to establish optimality. Suppose we want to minimize a nonlinear objective function $f(x)$ subject to equality constraints $h_j(x) = 0$ for $j = 1, \ldots, p$ and inequality constraints $g_i(x) \leq 0$ for $i = 1, \ldots, m$. The problem can be formulated as follows:
$$
\begin{align*}
    \min_{x\in\mathbb{R}^n} \; f(x)
    \quad\text{s.t.}\quad
    \begin{cases}
    g_i(x) \le 0, & i = 1,\dots,m,\\
    h_j(x) = 0, & j = 1,\dots,p,
    \end{cases}
\end{align*}
$$
where $x\in\mathbb{R}^{n}$ is the vector of decision variables, and the functions $f$, $g_i$, and $h_j$ have continuous first derivatives. A point that satisfies all of the constraints is called __feasible__. An inequality constraint is __active__ at $x$ when $g_i(x) = 0$, i.e., the point sits on that constraint's boundary.

### Lagrangian
For this problem, we introduce multipliers $\lambda_i\ge{0}$ for each inequality and $\nu_j$ (free, i.e., it can have either sign) for each equality. The Lagrangian is then given by:
$$
\boxed{
\begin{align*}
\mathcal L(x,\lambda,\nu) & =\;f(x)\;+\;\sum_{i=1}^m \lambda_i\,g_i(x)\;+\;\sum_{j=1}^p \nu_j\,h_j(x) \\
\lambda_i & \ge 0\;(i=1,\dots,m)\quad\text{convention}
\end{align*}}
$$
Each multiplier weights its constraint in the Lagrangian.

### Karush-Kuhn-Tucker (KKT) Conditions
The Karush–Kuhn–Tucker (KKT) conditions play a central role in the theory and practice of constrained nonlinear optimization by generalizing the method of Lagrange multipliers to handle both equality and inequality constraints. Assuming a suitable __constraint qualification__ (a regularity condition on the constraints, such as Slater's condition below), the following conditions are necessary for a point $x^*$ to be a (local) minimizer:

1. __Stationarity__: The gradient of the Lagrangian with respect to $x$ must vanish at the optimal point $x^*$:
    $$
    \begin{align*}
    \nabla_x\mathcal L(x^*,\lambda^*,\nu^*) = 0\quad\Longleftrightarrow\quad\nabla f(x^*) + \sum_{i=1}^{m} \lambda_i^* \nabla g_i(x^*) + \sum_{j=1}^{p} \nu_j^* \nabla h_j(x^*) = 0.
    \end{align*}
    $$
2. __Primal feasibility__: The constraints must be satisfied at the optimal point $x^*$:
    $$
    \begin{align*}
    & g_i(x^*) \le 0 \quad(i = 1, \ldots, m)\\
    & h_j(x^*) = 0 \quad(j = 1, \ldots, p).
    \end{align*}
    $$
3. __Dual feasibility__: The Lagrange multipliers for the inequality constraints must be non-negative:
    $$\lambda_i^* \ge 0 \quad(i = 1, \ldots, m).$$
4. __Complementary slackness__: For each inequality constraint, either the constraint is active (i.e., $g_i(x^*) = 0$) or the corresponding multiplier is zero ($\lambda_i^* = 0$):
    $$\lambda_i^* \cdot g_i(x^*) = 0 \quad(i = 1, \ldots, m).$$

These conditions provide a powerful framework for analyzing and solving constrained optimization problems. Which constraint qualification applies depends on the problem; for convex problems, the usual one is Slater's condition.

> __Slater's condition__: Suppose $f$ and each $g_i$ are convex and each $h_j$ is affine (linear plus a constant). Slater's condition is a constraint qualification for this convex problem that ensures the KKT conditions are necessary for optimality. It requires the existence of at least one point $x_0$ that strictly satisfies all inequality constraints and satisfies all equality constraints:
>
> $$g_i(x_0) < 0 \quad \text{for all } i = 1, \ldots, m \quad \text{and} \quad h_j(x_0) = 0 \quad \text{for all } j = 1, \ldots, p$$
>
> In other words, there exists a feasible point with slack in the inequality constraints.

__Interesting__: If $f$ and each $g_i$ are convex and each $h_j$ is affine, then any point satisfying the KKT conditions is a global minimizer! For a convex problem, the KKT conditions are therefore sufficient for optimality, and with Slater's condition they are also necessary. Convex functions were defined in the convexity box of the lecture.

### Example: One Variable, One Constraint
Let's see how the four conditions work on a small problem. Suppose we want to minimize $f(x) = (x-2)^{2}$ subject to the single inequality constraint $g(x) = x - c \leq 0$, where $c$ is a constant. The objective is convex and the constraint is affine, so the KKT conditions are sufficient. The Lagrangian is given by:
$$
\begin{equation*}
\mathcal{L}(x,\lambda) = (x-2)^{2} + \lambda\,(x-c)
\end{equation*}
$$
and stationarity requires $2(x^{\star}-2) + \lambda^{\star} = 0$. The answer depends on whether the constraint is active:

> __Cases:__
>
> * __Active constraint ($c=1$):__ If $\lambda^{\star}=0$, stationarity gives $x^{\star}=2$, which violates $x\leq 1$. So the multiplier is positive, complementary slackness makes the constraint active, $x^{\star}=1$, and stationarity gives $\lambda^{\star}=2\geq 0$. All four conditions hold, and the solution sits on the constraint boundary.
> * __Inactive constraint ($c=3$):__ The pair $x^{\star}=2$ and $\lambda^{\star}=0$ satisfies all four conditions, since $g(2) = -1 < 0$. The constraint has slack, so complementary slackness sets its multiplier to zero, and the solution is the minimum of $f$ without the constraint.

In the active case, the optimal value is $f(x^{\star}) = (c-2)^{2}$ for $c\leq 2$, which falls at the rate $2(2-c) = 2$ as $c$ increases from $c=1$. This rate equals $\lambda^{\star}=2$: in this example, the multiplier measures how fast loosening the constraint lowers the optimal value.

___

## Gradient Descent with Barrier and Penalty Terms
Given the general problem of constrained nonlinear optimization, we can apply _gradient descent_ to find a solution. Gradient descent is a first-order optimization algorithm, i.e., it uses only the gradient (the first derivatives) of the objective function, that iteratively updates the solution by moving in the direction of the steepest descent of the objective function. However, by default, gradient descent is designed for unconstrained optimization problems. Thus, we need to adapt it to handle constraints.

Toward this end, we can use barrier and penalty terms to incorporate the constraints into the objective function, as we did for simulated annealing in the lecture:

> __Penalty and barrier methods:__ A penalty term adds a cost that grows with the size of a constraint violation; we use a quadratic penalty for each equality constraint. A barrier term is finite inside the region where the inequality constraints hold and grows without bound as $x$ approaches the edge of that region; we use a logarithmic barrier for each inequality constraint, which keeps the solution strictly inside the region.

Consider the following (augmented) objective function that combines the original objective function with the barrier and penalty terms:
$$
\begin{align*}
    \min_{x\in\mathbb{R}^n}\;P_{\mu,\rho}(x)\;&=f(x)\;-\;\underbrace{\mu\sum_{i=1}^m\ln\bigl(-\,g_i(x)\bigr)}_{\text{barrier term}}\;+\;\underbrace{\frac{1}{2\rho}\sum_{j=1}^p
    \bigl[h_j(x)\bigr]^2}_{\text{penalty term}},\quad\text{where}\quad\mu>0,\;\rho>0
\end{align*}
$$

> __Parameters__
>
> The parameters $\mu$ and $\rho$ control the strength of these terms, and both are typically _decreased_ over iterations:
> * __Barrier weight__: As $\mu\to0$, the barrier term fades inside the region where the inequality constraints hold, so the solution can approach the edge of that region, where the optimum often lies. The barrier still grows without bound at the edge itself, so the solution never crosses it.
> * __Penalty weight__: As $\rho\to0$, the coefficient $\frac{1}{\rho}$ grows, strengthening the penalty effect and enforcing the equality constraints more strictly.
>
> We'll provide some heuristics for updating these parameters in the algorithm below.

### Algorithm
Let's develop a simple gradient descent algorithm for this problem. The algorithm iteratively updates the solution $x_k$ using the gradient of the augmented objective function $P_{\mu,\rho}(x)$.

__Initialization__: Given an initial guess $x_0$ that satisfies the inequality constraints strictly, i.e., $g_i(x_0)<0$ for all $i$, set $\mu > 0$ and $\rho > 0$. Specify a tolerance $\epsilon > 0$, a maximum number of iterations $K$, a step size (learning rate) $\alpha > 0$, and small final values $\mu_{\min}$ and $\rho_{\min}$ for the barrier and penalty weights. Set $\texttt{converged} \gets \texttt{false}$, the iteration counter to $k \gets 0$ and specify values for the penalty update parameters $\tau_{\mu},\tau_{\rho}\in\left(0,1\right)$.

While not $\texttt{converged}$ __do__:
1. Compute the gradient: $\nabla P_{\mu,\rho}(x_k) = \nabla f(x_k) + \mu \sum_{i=1}^m \frac{\nabla g_i(x_k)}{-g_i(x_k)} + \frac{1}{\rho} \sum_{j=1}^p h_j(x_k) \nabla h_j(x_k)$ evaluated at the current solution $x_k$.
2. Update the solution: $x_{k+1} = x_k - \alpha \nabla P_{\mu,\rho}(x_k)$.
    - If $g_i(x_{k+1})\geq 0$ for any $i$ (the barrier term is undefined there), or if $P_{\mu,\rho}(x_{k+1}) \geq P_{\mu,\rho}(x_k)$, __then__ halve the step size $\alpha$ and repeat this step.
    - $\texttt{Note}$: otherwise, $\alpha$ is fixed here, but it can be adapted dynamically based on the convergence behavior.
3. Check convergence:
     - If $\|\nabla P_{\mu,\rho}(x_{k+1})\|_{2} \leq \epsilon$, $|h_j(x_{k+1})|\leq\epsilon$ for all $j$, $\mu\leq\mu_{\min}$, and $\rho\leq\rho_{\min}$ __then__ set $\texttt{converged} \gets \texttt{true}$. Return $x_{k+1}$ as the approximate solution. A small gradient alone is not enough: while $\mu$ and $\rho$ are still large, the minimizer of $P_{\mu,\rho}$ can be far from the solution of the original problem. $\texttt{Note}$: here we look at the Euclidean norm of the gradient. However, other criteria can be used, such as the change in the solution or in the objective function value.
     - If $k+1 \geq K$ __then__ return $x_{k+1}$ and warn that the maximum number of iterations has been reached _without convergence_.
4. Increment the iteration counter: $k \gets k + 1$, update $\mu\gets \max(\tau_\mu\,\mu, \mu_{\min})$ and $\rho\gets \max(\tau_\rho\,\rho, \rho_{\min})$, and repeat.

As $\mu$ and $\rho$ decrease, the iterates can approach the edge of the region where the inequality constraints hold, and the equality constraints are enforced more strictly. Both changes can make the augmented objective change sharply over short distances, so the step size $\alpha$ may need to shrink as the iterations proceed.

___

## Beyond Basic Gradient Descent
The lecture's gradient descent algorithm uses a constant learning rate and only the gradient. Several techniques extend it:

> __Advanced optimization techniques:__
>
> * __Diminishing learning rates:__ Decrease the learning rate over time, e.g., $\alpha(k) = \alpha_{0}/(k+1)$ with an initial learning rate $\alpha_{0}>0$, when a good constant learning rate is unknown. Early iterations take larger steps, while later iterations make finer adjustments.
> * __Momentum methods:__ Build up velocity by accumulating gradients from previous iterations. This smooths oscillations and can speed up progress, especially along long, flat valleys of the objective.
> * __Second-order methods:__ Newton's method uses curvature information, the Hessian matrix of second derivatives, to take better-informed steps. When the objective has continuous second derivatives and the Hessian is positive definite near the minimum, it converges faster there, but it must compute the Hessian and solve a linear system at each step.

___

## Summary
In this notebook, we wrote the Lagrangian of the general constrained problem, stated the KKT conditions, adapted gradient descent to constrained problems with barrier and penalty terms, and looked at faster variants of gradient descent.

> __Key Takeaways:__
>
> * __KKT conditions:__ We introduced multipliers for the inequality and equality constraints and wrote the Lagrangian of the general problem. When a constraint qualification holds, a minimizer satisfies stationarity, primal feasibility, dual feasibility, and complementary slackness, and complementary slackness sets the multiplier of every inactive constraint to zero.
>
> * __Necessary and sufficient conditions:__ Slater's condition, a feasible point with slack in every inequality constraint, makes the KKT conditions necessary for a convex problem, and convexity makes them sufficient. In the one-variable example, we found the solution and its multiplier by checking the active and the inactive case in turn.
>
> * __Gradient descent with barrier and penalty terms:__ We folded the constraints into an augmented objective with a logarithmic barrier for the inequality constraints and a quadratic penalty for the equality constraints, and minimized it with gradient descent while the barrier and penalty weights decrease. Diminishing learning rates, momentum, and Newton's method extend this basic algorithm when a constant step is slow or hard to choose.

When the objective and the constraints are all linear, these conditions reduce to the KKT conditions for linear programs from week 5.

___